# LightGBM from scratch 

This notebook re-implements the ideas that make LightGBM fast, one at a time, on top of a small
gradient-boosting core (binary logistic loss, second-order / Newton leaf values):

1. **Histogram binning**: discretise every feature into <= `max_bin` bins, then search splits on bin
   boundaries instead of on every distinct raw value (compare with an exact greedy splitter).
2. **Histogram subtraction**: `hist(sibling) = hist(parent) - hist(child)`, so only the smaller child is scanned.
3. **Leaf-wise (best-first) growth** controlled by `num_leaves`, compared with level-wise growth.
4. **GOSS**: keep the large-gradient rows, sample the small-gradient rows and re-weight them.
5. **EFB**: bundle mutually exclusive sparse features (e.g. one-hot columns) into one column.

Everything is synthetic data, and the whole script runs in well under 90 seconds.

In [ ]:
import os, time, heapq
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('plots', exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25,
                     'axes.spines.top': False, 'axes.spines.right': False})
C = {'a': '#2a6f97', 'b': '#e07a1f', 'c': '#3a9d5d', 'd': '#9b4dca', 'g': '#777777'}
T0 = time.time()

## 1. Synthetic data and metrics

Binary classification with non-linear structure (a sine, an interaction, a threshold effect and an
absolute value) plus irrelevant noise columns. Two columns are made heavily skewed on purpose, since
that is where quantile binning matters.

In [ ]:
def make_data(n, n_feat=20, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, n_feat))
    X[:, 1] = np.exp(X[:, 1])            # log-normal (skewed)
    X[:, 6] = rng.exponential(2.0, n)    # exponential (skewed)
    logit = (2.0 * np.sin(1.5 * X[:, 0]) + 1.5 * X[:, 2] * X[:, 3]
             + 1.5 * (np.log(X[:, 1]) > 0.5) - 1.0 * np.abs(X[:, 4])
             + 0.8 * X[:, 5] + 0.4 * X[:, 6] - 0.3)
    y = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(float)
    return X, y


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def logloss(y, raw):
    p = np.clip(sigmoid(raw), 1e-12, 1 - 1e-12)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def auc(y, score):
    r = np.empty(len(y)); r[np.argsort(score, kind='stable')] = np.arange(1, len(y) + 1)
    n1 = y.sum(); n0 = len(y) - n1
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


X_all, y_all = make_data(30000)
Xtr, ytr, Xte, yte = X_all[:20000], y_all[:20000], X_all[20000:], y_all[20000:]
print('train', Xtr.shape, 'test', Xte.shape, 'positive rate %.3f' % ytr.mean())

## 2. Histogram binning

`Binner` learns up to `max_bin - 1` quantile edges per feature. Bin `b` holds values with
`edges[b-1] < x <= edges[b]`, so a split "bin <= b" is the same as the raw threshold "x <= edges[b]".
The result is a compact `uint8` matrix. Quantile (equal-frequency) bins put resolution where the data is,
which is why they are preferred over equal-width bins on skewed features.

In [ ]:
class Binner:
    def __init__(self, max_bin=255):
        self.max_bin = max_bin

    def fit(self, X):
        qs = np.linspace(0, 1, self.max_bin + 1)[1:-1]
        self.edges = [np.unique(np.quantile(X[:, j], qs)) for j in range(X.shape[1])]
        return self

    def transform(self, X):
        out = np.empty(X.shape, dtype=np.uint8)
        for j, e in enumerate(self.edges):
            out[:, j] = np.searchsorted(e, X[:, j], side='left')
        return out


fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
x = Xtr[:, 1]
xc = np.clip(x, None, np.quantile(x, 0.995))
ax[0].hist(xc, bins=80, color=C['g'], alpha=.6)
for e in Binner(16).fit(Xtr).edges[1]:
    ax[0].axvline(min(e, xc.max()), color=C['b'], lw=.8)
ax[0].set_title('Quantile edges (16 bins) on a skewed feature'); ax[0].set_xlabel('x1 (clipped at 99.5%)')
qb = Binner(16).fit(Xtr).transform(Xtr)[:, 1]
ub = np.clip(((x - x.min()) / (xc.max() - x.min()) * 16).astype(int), 0, 15)
ax[1].bar(np.arange(16) - .2, np.bincount(qb, minlength=16), .4, color=C['b'], label='quantile')
ax[1].bar(np.arange(16) + .2, np.bincount(ub, minlength=16), .4, color=C['a'], label='equal width')
ax[1].set_yscale('log'); ax[1].set_title('Rows per bin'); ax[1].set_xlabel('bin'); ax[1].legend()
nuniq = [len(np.unique(Xtr[:n, 0])) for n in (500, 2000, 8000, 20000)]
ax[2].plot([500, 2000, 8000, 20000], nuniq, 'o-', color=C['a'], label='exact: distinct thresholds')
ax[2].axhline(254, color=C['b'], label='hist: <= 254 thresholds')
ax[2].set_xscale('log'); ax[2].set_yscale('log'); ax[2].set_xlabel('rows'); ax[2].legend()
ax[2].set_title('Split candidates per feature')
plt.tight_layout(); plt.savefig('plots/01_binning.png'); plt.close()

## 3. The boosting core: histogram builder and split finders

Histogram layout is `(3, F, B)`: sum of gradients, sum of hessians and row count per (feature, bin).
Building it is one `np.bincount` over `bin + feature * B`. A split scan is then a `cumsum` over bins and a
vectorised gain evaluation for all `F * (B-1)` candidates at once:

$$\text{gain} = \tfrac12\Big[\tfrac{G_L^2}{H_L+\lambda} + \tfrac{G_R^2}{H_R+\lambda} - \tfrac{G^2}{H+\lambda}\Big]$$

The exact splitter sorts the node's rows for each feature and scans every distinct value.

In [ ]:
def build_hist(M, idx, g, h, B):
    """Gradient/hessian/count histogram of rows `idx`. Cost is O(len(idx) * F), independent of B."""
    n, F = len(idx), M.shape[1]
    flat = (M[idx] + np.arange(F) * B).ravel()
    w_g = np.repeat(g[idx], F); w_h = np.repeat(h[idx], F)
    size = F * B
    return np.stack([np.bincount(flat, w_g, size), np.bincount(flat, w_h, size),
                     np.bincount(flat, None, size).astype(float)]).reshape(3, F, B)


def best_split_hist(hist, lam, min_data, min_hess):
    g, h, c = hist
    G, H, N = g[0].sum(), h[0].sum(), c[0].sum()
    GL, HL, NL = np.cumsum(g, 1)[:, :-1], np.cumsum(h, 1)[:, :-1], np.cumsum(c, 1)[:, :-1]
    GR, HR, NR = G - GL, H - HL, N - NL
    gain = 0.5 * (GL**2 / (HL + lam) + GR**2 / (HR + lam) - G**2 / (H + lam))
    ok = (NL >= min_data) & (NR >= min_data) & (HL >= min_hess) & (HR >= min_hess)
    gain = np.where(ok, gain, -np.inf)
    k = int(np.argmax(gain)); f, b = divmod(k, gain.shape[1])
    return float(gain[f, b]), f, float(b)


def best_split_exact(Ms, g, h, lam, min_data, min_hess):
    n, F = Ms.shape
    G, H = g.sum(), h.sum(); base = G * G / (H + lam)
    best = (-np.inf, -1, 0.0); n_cand = 0
    NL = np.arange(1, n)
    for f in range(F):
        x = Ms[:, f]; o = np.argsort(x, kind='stable'); xs = x[o]
        GL = np.cumsum(g[o])[:-1]; HL = np.cumsum(h[o])[:-1]
        GR, HR = G - GL, H - HL
        gain = 0.5 * (GL**2 / (HL + lam) + GR**2 / (HR + lam) - base)
        ok = (xs[:-1] < xs[1:]) & (NL >= min_data) & (n - NL >= min_data) & (HL >= min_hess) & (HR >= min_hess)
        n_cand += int(ok.sum())
        if ok.any():
            gain = np.where(ok, gain, -np.inf); i = int(np.argmax(gain))
            if gain[i] > best[0]:
                best = (float(gain[i]), f, float(0.5 * (xs[i] + xs[i + 1])))
    return best + (n_cand,)

## 4. A GBDT with pluggable tree builder

One grower covers every variant we want to compare:

* `method='hist' | 'exact'`: how splits are searched.
* `growth='leaf' | 'level'`: which leaf is expanded next. Leaf-wise picks the leaf with the largest gain
  (a priority queue); level-wise expands leaves in breadth-first order. Both stop at `num_leaves`.
* `subtraction=True`: the larger child's histogram is `parent - smaller child`.
* `sampling=('goss', a, b)` or `('uniform', frac)`: row sampling per boosting round.

The model keeps counters (`rows_hist` = rows scanned while building histograms) so the savings can be
measured, not only timed.

In [ ]:
class GBDT:
    def __init__(self, method='hist', growth='leaf', num_leaves=31, max_depth=-1, lr=0.1, reg_lambda=1.0,
                 min_data=20, min_hess=1e-3, min_gain=1e-7, subtraction=True, sampling=None, seed=0):
        self.__dict__.update(locals()); del self.__dict__['self']

    # ---- one tree -------------------------------------------------------------------------
    def _grow(self, Ms, g, h, B):
        lam, lr = self.reg_lambda, self.lr
        feat, thr, left, right, val, depth_of = [], [], [], [], [], []

        def new_node(d):
            for l, v in ((feat, -1), (thr, 0.0), (left, -1), (right, -1), (val, 0.0), (depth_of, d)):
                l.append(v)
            return len(feat) - 1

        def eligible(n, d):
            return n >= 2 * self.min_data and (self.max_depth < 0 or d < self.max_depth)

        def hist_of(idx):
            self.stats['rows_hist'] += len(idx)
            return build_hist(Ms, idx, g, h, B)

        seq = [0]; gains = []

        def evaluate(node, idx, d, hist):
            val[node] = -g[idx].sum() / (h[idx].sum() + lam) * lr
            rec = dict(node=node, idx=idx, d=d, hist=hist, split=None, seq=seq[0]); seq[0] += 1
            if eligible(len(idx), d):
                if self.method == 'hist':
                    if hist is None:
                        rec['hist'] = hist_of(idx)
                    gain, f, t = best_split_hist(rec['hist'], lam, self.min_data, self.min_hess)
                    self.stats['candidates'] += Ms.shape[1] * (B - 1)
                else:
                    gain, f, t, nc = best_split_exact(Ms[idx], g[idx], h[idx], lam, self.min_data, self.min_hess)
                    self.stats['candidates'] += nc
                if gain > self.min_gain:
                    rec['split'] = (gain, f, t)
            return rec

        idx0 = np.arange(len(g)); root = new_node(0)
        cands, n_leaves = [evaluate(root, idx0, 0, None)], 1
        while n_leaves < self.num_leaves:
            cs = [r for r in cands if r['split']]
            if not cs:
                break
            if self.growth == 'leaf':
                rec = max(cs, key=lambda r: r['split'][0])          # best-first: largest gain anywhere
            else:
                rec = min(cs, key=lambda r: (r['d'], r['seq']))     # breadth-first
            cands.remove(rec)
            gain, f, t = rec['split']; idx = rec['idx']
            mask = Ms[idx, f] <= t
            li, ri = idx[mask], idx[~mask]
            ln, rn = new_node(rec['d'] + 1), new_node(rec['d'] + 1)
            feat[rec['node']], thr[rec['node']], left[rec['node']], right[rec['node']] = f, t, ln, rn
            gains.append(gain); n_leaves += 1
            d = rec['d'] + 1
            lh = rh = None
            if self.method == 'hist' and (eligible(len(li), d) or eligible(len(ri), d)):
                if self.subtraction:
                    if len(li) <= len(ri):
                        lh = hist_of(li); rh = rec['hist'] - lh
                    else:
                        rh = hist_of(ri); lh = rec['hist'] - rh
                else:
                    lh = hist_of(li) if eligible(len(li), d) else None
                    rh = hist_of(ri) if eligible(len(ri), d) else None
            rec['hist'] = None
            cands.append(evaluate(ln, li, d, lh)); cands.append(evaluate(rn, ri, d, rh))
        tree = dict(feat=np.array(feat), thr=np.array(thr), left=np.array(left), right=np.array(right),
                    val=np.array(val), depth=np.array(depth_of), gains=gains)
        tree['n_leaves'] = int((tree['feat'] < 0).sum())
        return tree

    @staticmethod
    def predict_tree(tree, M):
        node = np.zeros(len(M), dtype=np.int64)
        while True:
            f = tree['feat'][node]; rows = np.nonzero(f >= 0)[0]
            if len(rows) == 0:
                return tree['val'][node]
            nd = node[rows]
            go_left = M[rows, f[rows]] <= tree['thr'][nd]
            node[rows] = np.where(go_left, tree['left'][nd], tree['right'][nd])

    # ---- sampling -----------------------------------------------------------------------------
    def _sample(self, g, rng):
        n = len(g)
        if self.sampling is None:
            return None, 1.0
        if self.sampling[0] == 'uniform':
            return np.sort(rng.choice(n, int(self.sampling[1] * n), replace=False)), None
        _, a, b = self.sampling                                   # GOSS
        top_n, oth_n = int(a * n), int(b * n)
        order = np.argpartition(-np.abs(g), top_n)
        top, rest = order[:top_n], order[top_n:]
        pick = rng.choice(rest, oth_n, replace=False)
        idx = np.concatenate([top, pick])
        w = np.ones(len(idx)); w[top_n:] = (1 - a) / b             # amplify the sampled small-gradient rows
        return idx, w

    # ---- boosting loop --------------------------------------------------------------------------
    def fit(self, M, y, n_rounds, Mv=None, yv=None, B=None):
        rng = np.random.default_rng(self.seed)
        B = B or (int(M.max()) + 1 if self.method == 'hist' else 0)
        self.stats = dict(rows_hist=0, candidates=0, rows_used=[])
        self.trees, self.train_curve, self.val_curve = [], [], []
        p0 = np.clip(y.mean(), 1e-6, 1 - 1e-6); self.init = float(np.log(p0 / (1 - p0)))
        F = np.full(len(y), self.init); Fv = None if Mv is None else np.full(len(Mv), self.init)
        t0 = time.perf_counter()
        for _ in range(n_rounds):
            p = sigmoid(F); g, h = p - y, p * (1 - p)
            idx, w = self._sample(g, rng)
            if idx is None:
                Ms, gs, hs = M, g, h
            else:
                Ms, gs, hs = M[idx], g[idx], h[idx]
                if w is not None:
                    gs, hs = gs * w, hs * w
            self.stats['rows_used'].append(len(gs))
            tree = self._grow(Ms, gs, hs, B)
            self.trees.append(tree)
            F += self.predict_tree(tree, M)
            self.train_curve.append(logloss(y, F))
            if Mv is not None:
                Fv += self.predict_tree(tree, Mv); self.val_curve.append(logloss(yv, Fv))
        self.fit_time = time.perf_counter() - t0
        return self

    def decision(self, M, n_trees=None):
        F = np.full(len(M), self.init)
        for t in self.trees[:n_trees]:
            F += self.predict_tree(t, M)
        return F


ROUNDS, LR = 40, 0.1
binner = Binner(255).fit(Xtr)
Btr, Bte = binner.transform(Xtr), binner.transform(Xte)
print('binned dtype', Btr.dtype, 'bytes: raw %.1f MB -> binned %.1f MB' % (Xtr.nbytes / 1e6, Btr.nbytes / 1e6))

## 5. Histogram vs exact: accuracy and speed

Same algorithm, same hyper-parameters (31 leaves, 40 rounds), only the split search differs.
Then we sweep `max_bin` to see how coarse the histogram can be before accuracy suffers, and time one
root split search as the dataset grows.

In [ ]:
res = {}
for name, kw, (Mtr_, Mte_) in [('exact', dict(method='exact'), (Xtr, Xte)),
                               ('hist-255', dict(method='hist'), (Btr, Bte))]:
    m = GBDT(num_leaves=31, lr=LR, **kw).fit(Mtr_, ytr, ROUNDS, Mte_, yte)
    s = m.decision(Mte_)
    res[name] = m
    print('%-9s fit %.2fs  test logloss %.4f  AUC %.4f  candidates/round %d' %
          (name, m.fit_time, logloss(yte, s), auc(yte, s), m.stats['candidates'] // ROUNDS))
print('hist speed-up vs exact: %.1fx' % (res['exact'].fit_time / res['hist-255'].fit_time))

bins_list = [4, 8, 16, 32, 64, 255]
sweep = []
for mb in bins_list:
    bn = Binner(mb).fit(Xtr); a_, b_ = bn.transform(Xtr), bn.transform(Xte)
    m = GBDT(num_leaves=31, lr=LR).fit(a_, ytr, ROUNDS, b_, yte)
    sweep.append((m.val_curve[-1], m.fit_time))
    print('max_bin %3d  test logloss %.4f  fit %.2fs' % (mb, *sweep[-1]))

ns = [1000, 4000, 16000, 64000]; t_ex, t_hi = [], []
Xbig, ybig = make_data(max(ns), seed=5)
bnb = Binner(255).fit(Xbig); Bbig = bnb.transform(Xbig)
for n in ns:
    g = sigmoid(np.zeros(n)) - ybig[:n]; h = np.full(n, .25)
    t = time.perf_counter(); best_split_exact(Xbig[:n], g, h, 1, 20, 1e-3); t_ex.append(time.perf_counter() - t)
    t = time.perf_counter(); best_split_hist(build_hist(Bbig[:n], np.arange(n), g, h, 255), 1, 20, 1e-3)
    t_hi.append(time.perf_counter() - t)
print('root split search, n=%d: exact %.3fs vs hist %.3fs' % (ns[-1], t_ex[-1], t_hi[-1]))

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for k, (nm, col) in enumerate([('exact', C['a']), ('hist-255', C['b'])]):
    ax[0].plot(res[nm].val_curve, color=col, label=nm); ax[0].plot(res[nm].train_curve, color=col, ls=':')
ax[0].set_title('Logloss per round (solid test, dotted train)'); ax[0].set_xlabel('round'); ax[0].legend()
ax[1].plot(bins_list, [s[0] for s in sweep], 'o-', color=C['b'], label='hist')
ax[1].axhline(res['exact'].val_curve[-1], color=C['a'], ls='--', label='exact')
ax[1].set_xscale('log', base=2); ax[1].set_xlabel('max_bin'); ax[1].set_title('Test logloss vs max_bin'); ax[1].legend()
ax[2].loglog(ns, t_ex, 'o-', color=C['a'], label='exact (sort + scan)')
ax[2].loglog(ns, t_hi, 'o-', color=C['b'], label='histogram (build + scan)')
ax[2].set_xlabel('rows in node'); ax[2].set_ylabel('seconds'); ax[2].set_title('One root split search, 20 features'); ax[2].legend()
plt.tight_layout(); plt.savefig('plots/02_hist_vs_exact.png'); plt.close()

## 6. Histogram subtraction trick

For a split of a parent node into left and right children, `hist(parent) = hist(left) + hist(right)`
exactly. So only the **smaller** child needs to be scanned; the larger one is obtained by subtracting.
First we verify the identity numerically, then train with and without the trick: predictions must
match, but the number of rows scanned (and the time) drops.

In [ ]:
idx_parent = np.arange(5000); gg = sigmoid(np.zeros(20000)) - ytr; hh = np.full(20000, .25)
mask = Btr[idx_parent, 0] <= 120
li, ri = idx_parent[mask], idx_parent[~mask]
hp, hl = build_hist(Btr, idx_parent, gg, hh, 255), build_hist(Btr, li, gg, hh, 255)
hr_direct, hr_sub = build_hist(Btr, ri, gg, hh, 255), hp - hl
print('max |direct - subtracted| over all (stat, feature, bin): %.2e' % np.abs(hr_direct - hr_sub).max())

m_sub = GBDT(subtraction=True, lr=LR).fit(Btr, ytr, ROUNDS)
m_nos = GBDT(subtraction=False, lr=LR).fit(Btr, ytr, ROUNDS)
print('max prediction difference: %.2e' % np.abs(m_sub.decision(Bte) - m_nos.decision(Bte)).max())
print('rows scanned: no-subtraction %d, subtraction %d (%.0f%% fewer)' %
      (m_nos.stats['rows_hist'], m_sub.stats['rows_hist'], 100 * (1 - m_sub.stats['rows_hist'] / m_nos.stats['rows_hist'])))
print('fit time: %.2fs -> %.2fs' % (m_nos.fit_time, m_sub.fit_time))

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].bar(['no subtraction', 'subtraction'], [m_nos.stats['rows_hist'] / 1e6, m_sub.stats['rows_hist'] / 1e6], color=[C['g'], C['b']])
ax[0].set_ylabel('rows scanned (millions)'); ax[0].set_title('Histogram construction work')
ax[1].bar(['no subtraction', 'subtraction'], [m_nos.fit_time, m_sub.fit_time], color=[C['g'], C['b']])
ax[1].set_ylabel('seconds'); ax[1].set_title('Training time')
ax[2].plot(hr_direct[0, 0], label='right child, built directly', color=C['a'])
ax[2].plot(hr_sub[0, 0], '--', label='parent - left child', color=C['b'])
ax[2].set_title('Gradient histogram of feature 0'); ax[2].set_xlabel('bin'); ax[2].legend()
plt.tight_layout(); plt.savefig('plots/03_hist_subtraction.png'); plt.close()

## 7. Leaf-wise vs level-wise growth and `num_leaves`

With the same leaf budget, best-first growth spends every split where the gain is highest, so the
cumulative loss reduction of one tree is never below the breadth-first tree. The price is an unbalanced,
deeper tree that can overfit small data, which is why `num_leaves` is the main capacity knob.
We use a small training set (3000 rows) so the over-fitting regime is visible.

In [ ]:
one = {}
g0 = sigmoid(np.zeros(len(ytr))) - ytr; h0 = np.full(len(ytr), .25)
for gr in ('leaf', 'level'):
    gb = GBDT(growth=gr, num_leaves=16)
    one[gr] = gb._grow(Btr, g0, h0, int(Btr.max()) + 1) if hasattr(gb, 'stats') else None
    gb.stats = dict(rows_hist=0, candidates=0); one[gr] = gb._grow(Btr, g0, h0, 256)
for gr in one:
    lv = one[gr]['depth'][one[gr]['feat'] < 0]
    print('%-5s tree, 16 leaves: max depth %d, leaf depths %s, total gain %.1f' %
          (gr, lv.max(), sorted(lv.tolist()), sum(one[gr]['gains'])))

small = slice(0, 3000); leaves_list = [2, 4, 8, 16, 32, 64, 128]; R7 = {'leaf': [], 'level': []}
for gr in R7:
    for nl in leaves_list:
        m = GBDT(growth=gr, num_leaves=nl, lr=0.15, min_data=5).fit(Btr[small], ytr[small], 30, Bte, yte)
        R7[gr].append((m.train_curve[-1], m.val_curve[-1], max(t['depth'].max() for t in m.trees)))
print('num_leaves sweep (train/test logloss, max depth):')
for i, nl in enumerate(leaves_list):
    print('  leaves %3d | leaf-wise %.3f/%.3f d=%2d | level-wise %.3f/%.3f d=%2d' %
          (nl, *R7['leaf'][i], *R7['level'][i]))

fig, ax = plt.subplots(1, 4, figsize=(17, 3.6))
for gr, col in (('leaf', C['b']), ('level', C['a'])):
    gs_ = np.cumsum(one[gr]['gains']); ax[0].plot(range(1, len(gs_) + 1), gs_, 'o-', color=col, label=gr + '-wise')
    lv = one[gr]['depth'][one[gr]['feat'] < 0]
    ax[1].bar(np.arange(1, lv.max() + 1) + (.2 if gr == 'leaf' else -.2), np.bincount(lv, minlength=lv.max() + 1)[1:], .4, color=col, label=gr + '-wise')
    ax[2].plot(leaves_list, [r[0] for r in R7[gr]], 'o-', color=col, label=gr + '-wise')
    ax[3].plot(leaves_list, [r[1] for r in R7[gr]], 'o-', color=col, label=gr + '-wise')
ax[0].set_title('One tree: cumulative gain vs #splits'); ax[0].set_xlabel('splits'); ax[0].legend()
ax[1].set_title('Leaves per depth (16-leaf tree)'); ax[1].set_xlabel('depth'); ax[1].legend()
ax[2].set_title('Train logloss (3000 rows, 30 rounds)'); ax[2].set_xlabel('num_leaves'); ax[2].set_xscale('log', base=2); ax[2].legend()
ax[3].set_title('Test logloss'); ax[3].set_xlabel('num_leaves'); ax[3].set_xscale('log', base=2); ax[3].legend()
plt.tight_layout(); plt.savefig('plots/04_leafwise_vs_levelwise.png'); plt.close()

## 8. GOSS: Gradient-based One-Side Sampling

Keep the top `a` fraction of rows by `|gradient|`, draw a fraction `b` of the remaining rows at random,
and multiply the sampled small-gradient rows by `(1-a)/b` so sums stay unbiased. We check this two ways:

1. **Estimator quality**: for a fixed split, how well do GOSS and plain uniform sampling (same number of
   rows) estimate the split gain of the full data?
2. **Training**: accuracy and time versus full data and uniform subsampling.

In [ ]:
m_ref = GBDT(lr=LR).fit(Btr, ytr, 15)
Fcur = m_ref.decision(Btr); pcur = sigmoid(Fcur); g_c, h_c = pcur - ytr, pcur * (1 - pcur)
gain_full, f_s, b_s = best_split_hist(build_hist(Btr, np.arange(len(ytr)), g_c, h_c, 256), 1, 20, 1e-3)
left_mask = Btr[:, f_s] <= b_s


def gain_est(idx, w):
    gi, hi_ = g_c[idx] * w, h_c[idx] * w; lm = left_mask[idx]
    G, H = gi.sum(), hi_.sum(); GL, HL = gi[lm].sum(), hi_[lm].sum()
    return 0.5 * (GL**2 / (HL + 1) + (G - GL)**2 / (H - HL + 1) - G**2 / (H + 1))


rng = np.random.default_rng(1); a_, b_ = 0.2, 0.1; n = len(ytr)
est_goss, est_uni = [], []
for _ in range(400):
    top_n = int(a_ * n); order = np.argpartition(-np.abs(g_c), top_n)
    pick = rng.choice(order[top_n:], int(b_ * n), replace=False)
    w = np.concatenate([np.ones(top_n), np.full(len(pick), (1 - a_) / b_)])
    est_goss.append(gain_est(np.concatenate([order[:top_n], pick]), w))
    u = rng.choice(n, int((a_ + b_) * n), replace=False)
    est_uni.append(gain_est(u, np.full(len(u), 1 / (a_ + b_))))
est_goss, est_uni = np.array(est_goss), np.array(est_uni)
print('true split gain %.2f | GOSS mean %.2f std %.3f | uniform(30%%) mean %.2f std %.3f' %
      (gain_full, est_goss.mean(), est_goss.std(), est_uni.mean(), est_uni.std()))

cfgs = [('full data', None), ('uniform 30%', ('uniform', .3)), ('GOSS a=.2 b=.1 (30%)', ('goss', .2, .1)),
        ('uniform 15%', ('uniform', .15)), ('GOSS a=.1 b=.05 (15%)', ('goss', .1, .05))]
R8 = {}
for nm, sp in cfgs:
    m = GBDT(lr=LR, sampling=sp, seed=3).fit(Btr, ytr, 60, Bte, yte)
    R8[nm] = m; s = m.decision(Bte)
    print('%-24s rows/round %5d  fit %.2fs  test logloss %.4f  AUC %.4f' %
          (nm, np.mean(m.stats['rows_used']), m.fit_time, logloss(yte, s), auc(yte, s)))

fig, ax = plt.subplots(1, 4, figsize=(17, 3.6))
ag = np.sort(np.abs(g_c))[::-1]
ax[0].plot(ag, color=C['g']); ax[0].axvline(a_ * n, color=C['b'])
ax[0].fill_between(np.arange(int(a_ * n)), ag[:int(a_ * n)], color=C['b'], alpha=.5, label='kept: top 20%')
ax[0].fill_between(np.arange(int(a_ * n), n), ag[int(a_ * n):], color=C['a'], alpha=.3, label='sampled 10% + reweighted')
ax[0].set_title('Sorted |gradient| after 15 rounds'); ax[0].set_xlabel('rank'); ax[0].legend(fontsize=8)
bins_ = np.linspace(min(est_goss.min(), est_uni.min()), max(est_goss.max(), est_uni.max()), 40)
ax[1].hist(est_uni, bins_, alpha=.6, color=C['a'], label='uniform 30%'); ax[1].hist(est_goss, bins_, alpha=.6, color=C['b'], label='GOSS 20%+10%')
ax[1].axvline(gain_full, color='k', ls='--', label='full-data gain'); ax[1].set_title('Estimated gain of one split (400 draws)'); ax[1].legend(fontsize=8)
for (nm, _), col in zip(cfgs, [C['g'], C['a'], C['b'], '#8ab6d6', '#f0b67f']):
    ax[2].plot(R8[nm].val_curve, color=col, label=nm)
ax[2].set_title('Test logloss per round'); ax[2].set_xlabel('round'); ax[2].set_ylim(None, R8['full data'].val_curve[5]); ax[2].legend(fontsize=7)
ax[3].bar(range(len(cfgs)), [R8[nm].fit_time for nm, _ in cfgs], color=[C['g'], C['a'], C['b'], '#8ab6d6', '#f0b67f'])
ax[3].set_xticks(range(len(cfgs))); ax[3].set_xticklabels(['full', 'uni30', 'GOSS30', 'uni15', 'GOSS15']); ax[3].set_title('Fit time (s)')
plt.tight_layout(); plt.savefig('plots/05_goss.png'); plt.close()

## 9. EFB: Exclusive Feature Bundling

One-hot columns are almost never non-zero together. EFB greedily groups features whose non-zero rows
barely overlap (at most `max_conflict` shared rows), then merges each group into one column by giving
every member its own bin range (an offset). With zero conflicts the merge is lossless: from a bundle value
you can recover which feature was non-zero and its bin. Fewer columns means fewer histograms to build.

In [ ]:
def greedy_bundle(Xb, max_conflict=0, max_bins=255):
    nz = Xb > 0; order = np.argsort(-nz.sum(0)); bundles = []   # dense features first
    for f in order:
        for bd in bundles:
            if bd['bins'] + int(Xb[:, f].max()) <= max_bins and int((bd['mask'] & nz[:, f]).sum()) <= max_conflict:
                bd['feats'].append(int(f)); bd['mask'] |= nz[:, f]; bd['bins'] += int(Xb[:, f].max()); break
        else:
            bundles.append(dict(feats=[int(f)], mask=nz[:, f].copy(), bins=int(Xb[:, f].max())))
    return [bd['feats'] for bd in bundles]


def merge_bundles(Xb, bundles):
    out = np.zeros((len(Xb), len(bundles)), dtype=np.uint8)
    for j, fs in enumerate(bundles):
        off = 0
        for f in fs:
            r = Xb[:, f] > 0
            out[r, j] = Xb[r, f] + off; off += int(Xb[:, f].max())
    return out


def make_onehot_data(n, n_cat=6, levels=12, n_dense=6, seed=2):
    rng = np.random.default_rng(seed)
    cats = rng.integers(0, levels, size=(n, n_cat)); dense = rng.normal(size=(n, n_dense))
    eff = rng.normal(0, 1.0, size=(n_cat, levels))
    logit = sum(eff[c][cats[:, c]] for c in range(n_cat)) + dense[:, 0] + dense[:, 1] * dense[:, 2]
    X = np.hstack([dense] + [(cats[:, [c]] == np.arange(levels)).astype(float) for c in range(n_cat)])
    return X, (rng.random(n) < sigmoid(logit)).astype(float)


Xo, yo = make_onehot_data(24000); Xo_tr, yo_tr, Xo_te, yo_te = Xo[:16000], yo[:16000], Xo[16000:], yo[16000:]
bo = Binner(32).fit(Xo_tr); Bo_tr, Bo_te = bo.transform(Xo_tr), bo.transform(Xo_te)
bundles = greedy_bundle(Bo_tr, max_conflict=0)
Bb_tr, Bb_te = merge_bundles(Bo_tr, bundles), merge_bundles(Bo_te, bundles)
print('columns %d -> %d bundles; bundle sizes %s' % (Bo_tr.shape[1], len(bundles), sorted(len(b) for b in bundles)))
# losslessness check: decode one bundle and compare with the original columns
j = int(np.argmax([len(b) for b in bundles])); off = 0; ok = True
for f in bundles[j]:
    mx = int(Bo_tr[:, f].max()); col = np.where((Bb_tr[:, j] > off) & (Bb_tr[:, j] <= off + mx), Bb_tr[:, j] - off, 0)
    ok &= bool((col == Bo_tr[:, f]).all()); off += mx
print('bundle decodes back to original columns exactly:', ok)

rows = []
for nm, (a_tr, a_te) in [('original (%d cols)' % Bo_tr.shape[1], (Bo_tr, Bo_te)), ('EFB (%d cols)' % len(bundles), (Bb_tr, Bb_te))]:
    m = GBDT(lr=LR, num_leaves=31).fit(a_tr, yo_tr, 40, a_te, yo_te); s = m.decision(a_te)
    rows.append((nm, m.fit_time, logloss(yo_te, s), auc(yo_te, s)))
    print('%-20s fit %.2fs  test logloss %.4f  AUC %.4f' % rows[-1])

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].imshow(Bo_tr[:40] > 0, aspect='auto', cmap='Greys'); ax[0].set_title('Original: 78 sparse/dense columns'); ax[0].grid(False)
ax[1].imshow(Bb_tr[:40], aspect='auto', cmap='viridis'); ax[1].set_title('After EFB bundling'); ax[1].grid(False)
ax[2].bar([r[0] for r in rows], [r[1] for r in rows], color=[C['g'], C['b']]); ax[2].set_ylabel('fit seconds')
ax[2].set_title('Training time (test logloss %.3f vs %.3f)' % (rows[0][2], rows[1][2]))
plt.tight_layout(); plt.savefig('plots/06_efb.png'); plt.close()

## 10. Optional sanity check against a reference implementation

Only used to confirm the from-scratch models are in the right ballpark (skipped if unavailable).

In [ ]:
try:
    from sklearn.ensemble import HistGradientBoostingClassifier
    ref = HistGradientBoostingClassifier(max_iter=ROUNDS, learning_rate=LR, max_leaf_nodes=31, min_samples_leaf=20,
                                         early_stopping=False, random_state=0).fit(Xtr, ytr)
    pr = np.clip(ref.predict_proba(Xte)[:, 1], 1e-12, 1 - 1e-12)
    print('sklearn HistGB test logloss %.4f vs ours %.4f' % (-np.mean(yte * np.log(pr) + (1 - yte) * np.log(1 - pr)), res['hist-255'].val_curve[-1]))
except Exception as e:
    print('sklearn not available, skipped:', type(e).__name__)
print('total runtime %.1fs' % (time.time() - T0))

## Take-aways

* Histograms turn split search from "sort + scan all distinct values" into "bincount + cumsum over <= 255 bins";
  accuracy is essentially unchanged for `max_bin` >= ~32 and the speed-up grows with the number of rows.
* Subtraction gives identical trees while scanning roughly half the rows.
* Leaf-wise growth reaches a lower training loss per leaf than level-wise; unconstrained it overfits small data,
  so tune `num_leaves` (and `min_data_in_leaf`) first.
* GOSS keeps the gradient signal of hard rows; with the same sampled fraction its gain estimates are closer to
  the full-data value than uniform sampling when gradients are skewed.
* EFB shrinks the number of columns of one-hot data without losing information.